# Cell cycle in morphology and RNA: the Element Cytos multiomics screen

The Element Cytos assay reads Cell Painting, four protein stains, the CRISPR guide and 3' RNA in the
**same cells**. That lets us ask a question most morphology data cannot: does a cell-cycle clock learned
from the RNA of a cell agree with the morphology of that same cell, and is the cell cycle something to
validate against or to correct for in a perturbation screen?

The screen is a six-gene knockout panel in A549 cells (CTNNB1, GSK3B, IL1R1, OR1I1, SMAD7, TGFBR2) with a
non-targeting control (NTC), split across TNF-alpha stimulation and baseline. OR1I1 is an olfactory
receptor that A549 cells do not express, so it should behave like a control, although nothing below uses it
as one. We load the staged object, whose RNA cell-cycle
clock was fitted once with CycleVI and stored on the cells, so nothing here retrains a model.

This notebook is also a test of the mantispy tooling built for this analysis. Most sections use one of the
new functions on real data, and the last section lists what each one did.

In [ ]:
%matplotlib inline
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pooch
import scanpy as sc

import mudata as md
import mantispy as mt

URL = "https://exampledata.scverse.org/sbi2-cytodata/multiomics/cytos_multiomics_cycle_v1.h5mu"
SHA = "sha256:1ea59dec251c2926c33379cbb0ad14bef688aad4b32a90175e1ecd5cb2282230"
path = pooch.retrieve(
    URL, known_hash=SHA, fname="cytos_multiomics_cycle_v1.h5mu", path=pooch.os_cache("mantispy-casestudies")
)

mdata = md.read(path)
obs = mdata.obs
# The imaging modality is already a stamped mantispy object; the shared metadata lives on the MuData.
img = mdata["imaging"].copy()
img.obs = obs.loc[img.obs_names].copy()
# The CycleVI RNA clock, in radians, wrapped to [-pi, pi).
img.obs["clock"] = np.deg2rad((img.obs["cycle_position_deg"].to_numpy() + 180) % 360 - 180)
print(mdata)
print("cells per knockout:", obs["target_gene"].value_counts().to_dict())
print("TNF-alpha conditions per plate and per well:")
print(obs.groupby(["Metadata_Plate", "Metadata_Well"], observed=True)["condition"].unique())

## The object is ordinary AnnData, so scanpy works on it

mantispy stores profiles on AnnData and does not reimplement what scanpy already does. After the selected
features are scaled and their missing values filled, the imaging object goes into `sc.pp.pca`, `sc.pp.neighbors`, `sc.tl.umap` and `sc.tl.leiden`. We run them on a
sample of the cells to keep the embedding quick.

In [ ]:
sub = sc.pp.subsample(img, n_obs=30000, copy=True, random_state=0)
sub = sub[:, sub.var["selected"].to_numpy()].copy()  # the features feature-selection keeps
sub.X = np.nan_to_num(np.asarray(sub.X, dtype="float32"))
sc.pp.scale(sub, max_value=10)
sc.pp.pca(sub, n_comps=30)
sc.pp.neighbors(sub, n_neighbors=15)
sc.tl.umap(sub)
sc.tl.leiden(sub, flavor="igraph", n_iterations=2, directed=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
sc.pl.umap(sub, color="target_gene", ax=axes[0], show=False, title="perturbation", size=6)
sc.pl.umap(
    sub, color="cycle_position_deg", ax=axes[1], show=False, title="CycleVI clock (deg)", size=6, cmap="twilight"
)
fig.tight_layout()
plt.show()
print("leiden clusters:", sub.obs["leiden"].nunique())

## Two readings of the cell cycle

mantispy can call a discrete phase from the integrated DNA intensity. The fit assumes a G1 peak and a G2M
peak one doubling apart. On this data the DNA stain has one main peak in every well, and
`cell_cycle_phase` raises one warning that names those wells. It still returns a split, and the split is
not usable here: it calls about three quarters of the cells S phase. The
fit runs per well, because staining intensity is not comparable across wells. To see what the discrete
call is worth here, the share of each phase is plotted along the RNA clock: a working DNA call would put
G1 and G2M cells at different clock positions.

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    mt.tl.cell_cycle_phase(img, dna_feature="Intensity_IntegratedIntensity_Nucleus", layer="raw", by="Metadata_Well")
for w in caught:
    if "one DNA peak" in str(w.message):
        print("warning:", w.message)
print("discrete phase calls:", img.obs["Metadata_CellCyclePhase"].value_counts(normalize=True).round(3).to_dict())

bins = np.linspace(-np.pi, np.pi, 25)
centre = np.rad2deg((bins[:-1] + bins[1:]) / 2)
which = np.digitize(img.obs["clock"].to_numpy(), bins) - 1
phase = img.obs["Metadata_CellCyclePhase"].astype(str).to_numpy()
fig, ax = plt.subplots(figsize=(5.2, 3.2))
for name in ("G1", "S", "G2M"):
    share = [np.mean(phase[which == b] == name) if (which == b).any() else np.nan for b in range(len(centre))]
    ax.plot(centre, share, marker="o", ms=3, label=name)
ax.set_xlabel("RNA clock (degrees)")
ax.set_ylabel("share of cells")
ax.set_title("DNA phase call along the RNA clock")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## Morphology follows the RNA clock weakly but reproducibly

The discrete DNA call is weak here, but the question is whether the morphology follows the RNA clock at
all. `feature_cycle_association` fits each imaging feature to the clock within each well and writes the
share of its variance the clock explains, with a split-half check that the fitted curve is reproducible.
We compare its ranking against the hand-built feature ranking that shipped with the dataset.

The clock explains at most about 11% of any single feature's variance within a well, so it is a small part
of the morphology. The fitted curves are nevertheless reproducible, with split-half correlations above 0.99,
and the top features all measure the nuclear DNA stain.

In [ ]:
mt.tl.feature_cycle_association(img, angle_key="clock", by="Metadata_Well")
assoc = img.uns["mantispy"]["feature_cycle_association"]
cmp = assoc.set_index("feature").join(img.var["clock_r2"]).dropna(subset=["clock_r2"])
rho = cmp["r2"].corr(cmp["clock_r2"], method="spearman")
print(f"Spearman rho against the hand-built ranking: {rho:.3f}")

fig, ax = plt.subplots(figsize=(4.6, 4.3))
ax.scatter(cmp["clock_r2"], cmp["r2"], s=5, alpha=0.4)
ax.set_xlabel("hand-built clock r2 (shipped with the data)")
ax.set_ylabel("mt.tl.feature_cycle_association r2")
ax.set_title(f"Spearman rho = {rho:.3f}")
fig.tight_layout()
plt.show()
assoc.nlargest(5, "r2")[["feature", "r2", "split_half_r"]]

## The cycle in the RNA embeddings

`circular_neighbor_error` asks how well an embedding places cells next to cells at the same clock position.


In [ ]:
# How well does an embedding line cells up by cell-cycle position? Lower angular error is better.
from sklearn.neighbors import NearestNeighbors

rng = np.random.default_rng(0)
s = rng.choice(img.n_obs, 30000, replace=False)
theta = img.obs["clock"].to_numpy()[s]
for name in ("X_cyclevi_cycle", "X_drvi"):
    emb = np.asarray(mdata["rna"].obsm[name])[s]
    idx = NearestNeighbors(n_neighbors=16).fit(emb).kneighbors(return_distance=False)[:, 1:]
    print(f"circular_neighbor_error on {name}: {mt.tl.circular_neighbor_error(theta, idx):.3f}")
random_idx = rng.integers(0, len(s), size=idx.shape)
print(f"circular_neighbor_error with random neighbours: {mt.tl.circular_neighbor_error(theta, random_idx):.3f}")
# X_cyclevi_cycle is the clock's own two-dimensional latent, so its error is near zero by construction.
# X_drvi is an RNA embedding learned without the clock, and random neighbours give the error of an embedding
# that carries no cycle at all.

## The knockouts in the RNA latent space

The DrVI embedding is a latent of the RNA, not of the morphology. `io.from_embedding` wraps it into a
tool-ready object in one call, which `tl.aggregate` and `tl.map` then accept directly. `map` in activity
mode scores how far each knockout's profiles sit from the non-targeting cells in that RNA space.

Only CTNNB1 and TGFBR2 separate from NTC; GSK3B, IL1R1, SMAD7 and the OR1I1 control do not.

In [ ]:
drvi = mt.io.from_embedding(np.asarray(mdata["rna"].obsm["X_drvi"]), obs, resolution="object")
agg = mt.tl.aggregate(drvi, by=("Metadata_Plate", "Metadata_Well", "Metadata_Perturbation"), min_cells=50)
mt.pp.annotate_controls(agg, negcon=["NTC"], perturbation_key="Metadata_Perturbation")
mt.tl.map(agg, mode="activity", reference="negcon", batch="Metadata_Plate", null_size=2000)

hits = agg.uns["mantispy"]["map"].sort_values("score", ascending=False)
fig, ax = plt.subplots(figsize=(5.2, 3.2))
ax.barh(
    hits["Metadata_Perturbation"], hits["score"], color=["#2b8cbe" if s else "#cccccc" for s in hits["is_significant"]]
)
ax.invert_yaxis()
ax.set_xlabel("activity mAP score")
ax.set_title("separation from NTC in the RNA latent (blue = significant)")
fig.tight_layout()
plt.show()
hits[["Metadata_Perturbation", "score", "qvalue", "is_significant"]]

## What moves the morphology

`batch_variance_explained` puts the TNF-alpha condition, the knockout and the circular clock on one scale:
the share of the principal-component variance each explains. Every plate and well holds a single condition
(printed above), so the features are normalized once against all NTC cells rather than per plate or well.

The condition explains about 10% of the variance, the clock about 0.6% and the knockout about 0.15%.


In [ ]:
# Normalizing or centring per plate or per well would subtract the TNF-alpha condition itself.
imn = img.copy()
imn.obs["is_ntc"] = imn.obs["Metadata_Perturbation"].astype(str).eq("NTC").to_numpy()
mt.pp.normalize(imn, layer="raw", by=None, reference="is_ntc")
sel = imn.var["selected"].to_numpy() & ~imn.var["degenerate_scale"].to_numpy()

# How much of the morphology variance is the cell cycle, next to TNF-alpha and the knockout, on one scale?
imp = imn[:, sel].copy()
imp.X = np.nan_to_num(np.asarray(imp.X, dtype="float32"))
sc.pp.pca(imp, n_comps=30)
bv = mt.metrics.batch_variance_explained(
    imp, keys=["condition", "Metadata_Perturbation"], circular=["clock"], use_rep="X_pca"
)
fig, ax = plt.subplots(figsize=(4.4, 3.0))
ax.bar(bv["key"], bv["value"], color=["#756bb1", "#998ec3", "#f1a340"])
ax.set_ylabel("share of PC variance")
ax.set_title("what moves the morphology")
plt.xticks(rotation=15)
fig.tight_layout()
plt.show()
bv

## Does a knockout move cells around the cycle?

The clock is an angle, so a knockout can move cells around the cycle. `circular_shift` measures that
shift against the non-targeting cells within each well. Its p-value comes from shuffling guides between
the knockout and the control within each well, so the guide, not the cell, is the replicate. The test is
on the mean position, so it can miss a knockout that empties one arc and fills the opposite one.

Only CTNNB1 moves cells, by about 17 degrees toward smaller clock angles. The other knockouts and the
OR1I1 control do not.

In [ ]:
# Only cells with an assigned guide can be shuffled as part of one.
assigned = img[img.obs["guide_assigned"].astype(bool).to_numpy()].copy()
mt.tl.circular_shift(
    assigned,
    angle_key="clock",
    by="Metadata_Perturbation",
    reference="NTC",
    block="Metadata_Well",
    unit="guide",
    n_permutations=999,
)
shift = assigned.uns["mantispy"]["circular_shift"].sort_values("shift")
fig, ax = plt.subplots(figsize=(5.0, 3.2))
ax.barh(
    shift["group"],
    np.rad2deg(shift["shift"]),
    color=["#d7301f" if q < 0.05 else "#cccccc" for q in shift["qvalue"].fillna(1.0)],
)
ax.axvline(0, color="k", lw=0.6)
ax.set_xlabel("circular shift vs NTC (degrees)")
ax.set_title("does the knockout move cells around the cycle? (red: q < 0.05)")
fig.tight_layout()
plt.show()
shift

## What each function did

Each function below did a job that the hackathon notebooks did by hand.

| function | what it did here | what it replaces |
|---|---|---|
| `io.from_embedding` | wrapped the DrVI RNA latent into a tool-ready object for `aggregate` and `map` | the stamp-an-embedding boilerplate repeated in every hackathon notebook |
| `tl.feature_cycle_association` | ranked imaging features by the RNA clock, with a split-half check | a hand-written within-well least squares on sine and cosine |
| `tl.cell_cycle_phase` | called phases per well and warned that the DNA has one peak, so its split is not usable here | a silent and unreliable three-bin call |
| `tl.circular_neighbor_error` | scored how well each embedding lines cells up by clock position | a hand-written circular neighbour statistic |
| `metrics.batch_variance_explained(circular=…)` | put the clock on one variance scale with TNF-alpha and the knockout | passing sine and cosine as two separate covariates |
| `tl.circular_shift` | tested which knockouts move cells around the cycle, with the guide as the unit | a hand-written per-well circular-mean test |